# Temporal phase-classification model — research evaluation

This notebook trains and evaluates a configurable martial-arts temporal
phase classifier. The jab is the current representative evaluation
technique, not the definition or limit of the framework.

It uses human-verified session annotations for validation/test, splits
groups before windowing, compares non-learned baselines, reports frame and
boundary metrics, and checks ONNX deployment parity.

In [ ]:
!pip -q install onnx onnxruntime scikit-learn pandas seaborn

In [ ]:
import hashlib, json, os, platform, random, time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "PyTorch:", torch.__version__)

## 1. Configuration and phase vocabulary

Adjust the phase list only before annotation/training. The labels below are
a framework vocabulary; a technique-specific mapping may use a subset.

In [ ]:
@dataclass
class Config:
    data_path: str = "/content/phase_sessions.json"
    output_root: str = "/content/research_outputs/phase_classifier"
    technique_id: str = "jab"
    window: int = 90
    stride: int = 15
    joints: int = 33
    features: int = 4
    batch_size: int = 32
    epochs: int = 60
    patience: int = 10
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    hidden_dim: int = 96
    dropout: float = 0.15
    test_fraction: float = 0.20
    validation_fraction_of_remaining: float = 0.25
    seeds: tuple = (42, 43, 44)
    boundary_tolerance_frames: int = 5

CFG = Config()
PHASES = [
    "__PAD__", "__UNKNOWN__", "__TRACKING_LOST__",
    "PREPARATION", "ENTRY", "EXECUTION", "PEAK",
    "RETRACTION", "RECOVERY"
]
PHASE_TO_ID = {name: index for index, name in enumerate(PHASES)}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path(CFG.output_root) / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(asdict(CFG), PHASE_TO_ID)

## 2. Strict session loader and normalization

Expected JSON is documented in `research/data/README.md`. Validation and
test sessions must have `annotation_status = human_verified`. Synthetic
data, if included, is restricted to training and remains explicitly marked.

In [ ]:
def sha256_file(path, block_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize_landmarks(values):
    values = np.asarray(values, dtype=np.float32)
    if values.shape != (33, 4):
        raise ValueError(f"Expected [33,4], received {values.shape}")
    xyz, visibility = values[:, :3].copy(), values[:, 3:4].copy()
    center = (xyz[23] + xyz[24]) / 2
    shoulder_width = np.linalg.norm(xyz[11] - xyz[12])
    scale = max(float(shoulder_width), 1e-6)
    xyz = (xyz - center) / scale
    return np.concatenate((xyz, np.clip(visibility, 0, 1)), axis=-1)

def labels_from_segments(frame_count, segments):
    labels = np.full(frame_count, PHASE_TO_ID["__UNKNOWN__"], dtype=np.int64)
    for segment in segments:
        phase = str(segment["phase"]).upper()
        if phase not in PHASE_TO_ID:
            raise ValueError(f"Unknown phase: {phase}")
        start, end = int(segment["start_frame"]), int(segment["end_frame"])
        if not (0 <= start < end <= frame_count):
            raise ValueError(f"Invalid segment [{start},{end})")
        labels[start:end] = PHASE_TO_ID[phase]
    return labels

def load_sessions(path, technique_id):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    records = payload["sessions"] if isinstance(payload, dict) else payload
    sessions = []
    for record in records:
        if record.get("technique_id") != technique_id:
            continue
        frames = record["frames"]
        landmarks = np.stack([normalize_landmarks(frame["landmarks"]) for frame in frames])
        labels = labels_from_segments(len(frames), record["segments"])
        sessions.append({
            "session_id": str(record["session_id"]),
            "participant_id": str(record["participant_id"]),
            "fps": float(record.get("fps", 30)),
            "annotation_status": str(record.get("annotation_status", "unverified")),
            "source_type": str(record.get("source_type", "real")),
            "x": landmarks,
            "y": labels,
        })
    if not sessions:
        raise ValueError(f"No sessions found for technique_id={technique_id!r}")
    return sessions

sessions = load_sessions(CFG.data_path, CFG.technique_id)
print("Sessions:", len(sessions), "Participants:", len({s["participant_id"] for s in sessions}))
print("Dataset SHA-256:", sha256_file(CFG.data_path))

## 3. Grouped split before windowing

Real sessions are split first. Synthetic sessions can augment only the
training split. Windows inherit their session split, preventing overlap
leakage.

In [ ]:
def split_real_sessions(sessions, cfg, seed):
    real_indices = np.array([
        i for i, session in enumerate(sessions) if session["source_type"] != "synthetic"
    ])
    participants = np.array([sessions[i]["participant_id"] for i in real_indices])
    session_ids = np.array([sessions[i]["session_id"] for i in real_indices])
    groups = participants if len(np.unique(participants)) >= 3 else session_ids
    group_level = "participant" if groups is participants else "session"
    outer = GroupShuffleSplit(n_splits=1, test_size=cfg.test_fraction, random_state=seed)
    train_val_rel, test_rel = next(outer.split(real_indices, groups=groups))
    train_val = real_indices[train_val_rel]
    test = real_indices[test_rel]
    inner_groups = groups[train_val_rel]
    inner = GroupShuffleSplit(
        n_splits=1, test_size=cfg.validation_fraction_of_remaining,
        random_state=seed + 1
    )
    train_rel, val_rel = next(inner.split(train_val, groups=inner_groups))
    train, val = train_val[train_rel], train_val[val_rel]
    synthetic = np.array([
        i for i, session in enumerate(sessions) if session["source_type"] == "synthetic"
    ], dtype=int)
    train = np.concatenate((train, synthetic))
    for index in np.concatenate((val, test)):
        if sessions[index]["annotation_status"] != "human_verified":
            raise ValueError("Validation/test sessions must be human_verified.")
    return {"train": train, "validation": val, "test": test}, group_level

def create_windows(indices, sessions, cfg):
    xs, ys, rows = [], [], []
    for session_index in indices:
        session = sessions[session_index]
        if len(session["x"]) < cfg.window:
            continue
        for start in range(0, len(session["x"]) - cfg.window + 1, cfg.stride):
            end = start + cfg.window
            xs.append(session["x"][start:end])
            ys.append(session["y"][start:end])
            rows.append({
                "session_index": int(session_index),
                "session_id": session["session_id"],
                "participant_id": session["participant_id"],
                "start_frame": start,
                "fps": session["fps"],
                "source_type": session["source_type"],
            })
    if not xs:
        raise ValueError("A split produced no windows.")
    return np.stack(xs), np.stack(ys), pd.DataFrame(rows)

split_indices, GROUP_LEVEL = split_real_sessions(sessions, CFG, CFG.seeds[0])
split_data = {
    name: create_windows(indices, sessions, CFG)
    for name, indices in split_indices.items()
}
for name, (x, y, rows) in split_data.items():
    print(name, x.shape, "sessions:", rows.session_id.nunique())

## 4. Spatial-graph and temporal-convolution classifier

The model produces one phase label per input frame. Its graph mixing is
based on MediaPipe adjacency and its temporal layers use dilated
convolutions. This notebook calls it a temporal phase classifier; it does
not claim universal technique coverage.

In [ ]:
EDGES = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(25,27),(27,29),(29,31),
    (27,31),(24,26),(26,28),(28,30),(30,32),(28,32)
]

def normalized_adjacency(joints=33):
    adjacency = torch.eye(joints)
    for a, b in EDGES:
        adjacency[a,b] = adjacency[b,a] = 1
    degree = adjacency.sum(1).clamp_min(1)
    inv_sqrt = degree.pow(-0.5)
    return inv_sqrt[:,None] * adjacency * inv_sqrt[None,:]

class PhaseDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, index):
        return self.x[index], self.y[index]

class SpatialGraphBlock(nn.Module):
    def __init__(self, input_dim, output_dim, dropout):
        super().__init__()
        self.self_projection = nn.Linear(input_dim, output_dim)
        self.neighbor_projection = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("adjacency", normalized_adjacency())
    def forward(self, x):
        neighbors = torch.einsum("jk,btkd->btjd", self.adjacency, x)
        return self.norm(F.gelu(
            self.self_projection(x) + self.neighbor_projection(neighbors)
        ))

class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        padding = dilation
        self.conv = nn.Conv1d(
            channels, channels, kernel_size=3,
            padding=padding, dilation=dilation
        )
        self.norm = nn.BatchNorm1d(channels)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return x + self.dropout(F.gelu(self.norm(self.conv(x))))

class TemporalPhaseClassifier(nn.Module):
    def __init__(self, cfg, class_count):
        super().__init__()
        self.spatial1 = SpatialGraphBlock(cfg.features, cfg.hidden_dim, cfg.dropout)
        self.spatial2 = SpatialGraphBlock(cfg.hidden_dim, cfg.hidden_dim, cfg.dropout)
        self.temporal = nn.Sequential(*[
            TemporalResidualBlock(cfg.hidden_dim, dilation, cfg.dropout)
            for dilation in (1, 2, 4, 8)
        ])
        self.classifier = nn.Conv1d(cfg.hidden_dim, class_count, 1)
    def forward(self, x):
        x = self.spatial2(self.spatial1(x)).mean(dim=2)
        return self.classifier(self.temporal(x.transpose(1,2))).transpose(1,2)

## 5. Weighted training and early stopping

Class weights are calculated from training frames only. Validation macro
F1 selects the checkpoint; test labels remain unused during training.

In [ ]:
def class_weights(labels, class_count):
    counts = np.bincount(labels.reshape(-1), minlength=class_count).astype(float)
    weights = counts.sum() / np.maximum(counts, 1)
    weights = weights / weights.mean()
    return torch.as_tensor(weights, dtype=torch.float32)

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    true, predicted, probabilities = [], [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        probabilities.append(torch.softmax(logits, -1).cpu().numpy())
        predicted.append(logits.argmax(-1).cpu().numpy())
        true.append(y.numpy())
    return np.concatenate(true), np.concatenate(predicted), np.concatenate(probabilities)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_one_seed(seed, train_arrays, val_arrays, cfg):
    seed_everything(seed)
    train_loader = DataLoader(
        PhaseDataset(*train_arrays), batch_size=cfg.batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )
    val_loader = DataLoader(
        PhaseDataset(*val_arrays), batch_size=cfg.batch_size, shuffle=False
    )
    model = TemporalPhaseClassifier(cfg, len(PHASES)).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    weights = class_weights(train_arrays[1], len(PHASES)).to(DEVICE)
    best, state, wait, history = -1.0, None, 0, []
    for epoch in range(1, cfg.epochs + 1):
        model.train(); losses = []
        for x, y in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(x.to(DEVICE))
            loss = F.cross_entropy(
                logits.reshape(-1, len(PHASES)), y.to(DEVICE).reshape(-1),
                weight=weights
            )
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); losses.append(loss.item())
        true, predicted, _ = collect_predictions(model, val_loader)
        score = f1_score(true.ravel(), predicted.ravel(), average="macro", zero_division=0)
        history.append({"epoch": epoch, "train_loss": np.mean(losses), "val_macro_f1": score})
        print(f"seed={seed} epoch={epoch:03d} loss={np.mean(losses):.5f} val_f1={score:.5f}")
        if score > best:
            best, wait = score, 0
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if wait >= cfg.patience:
            break
    model.load_state_dict(state)
    return model, pd.DataFrame(history)

## 6. Frame metrics and transition-boundary metrics

Accuracy alone can hide poor minority-phase behavior, so macro F1,
balanced accuracy, per-class scores, and confusion matrices are required.
Boundary matching reports precision/recall and frame timing error within a
declared tolerance.

In [ ]:
def frame_metrics(true, predicted):
    true, predicted = true.ravel(), predicted.ravel()
    return {
        "accuracy": float(accuracy_score(true, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(true, predicted)),
        "macro_f1": float(f1_score(true, predicted, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(true, predicted, average="weighted", zero_division=0)),
        "report": classification_report(
            true, predicted, labels=np.arange(len(PHASES)),
            target_names=PHASES, output_dict=True, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            true, predicted, labels=np.arange(len(PHASES))
        ).tolist(),
    }

def transition_indices(labels):
    return np.flatnonzero(labels[1:] != labels[:-1]) + 1

def boundary_metrics(true, predicted, tolerance):
    errors, matched_pred = [], set()
    true_boundaries, pred_boundaries = transition_indices(true), transition_indices(predicted)
    for boundary in true_boundaries:
        candidates = [
            (abs(int(other) - int(boundary)), index)
            for index, other in enumerate(pred_boundaries)
            if index not in matched_pred and abs(int(other) - int(boundary)) <= tolerance
        ]
        if candidates:
            error, index = min(candidates)
            errors.append(error); matched_pred.add(index)
    matches = len(errors)
    return {
        "boundary_precision": matches / max(len(pred_boundaries), 1),
        "boundary_recall": matches / max(len(true_boundaries), 1),
        "boundary_mae_frames": float(np.mean(errors)) if errors else None,
        "true_boundary_count": int(len(true_boundaries)),
        "predicted_boundary_count": int(len(pred_boundaries)),
    }

def majority_baseline(train_y, shape):
    majority = int(np.bincount(train_y.ravel(), minlength=len(PHASES)).argmax())
    return np.full(shape, majority, dtype=np.int64)

## 7. Repeated held-out evaluation

Window-level frame scores are supplemented with per-window boundary
results. For the final thesis, reconstruct full session timelines (average
overlapping probabilities) and have the deployed repetition decoder scored
against repetition annotations in `pilot-study/phase_annotations`.

In [ ]:
train_x, train_y, _ = split_data["train"]
val_x, val_y, _ = split_data["validation"]
test_x, test_y, test_rows = split_data["test"]
test_loader = DataLoader(PhaseDataset(test_x, test_y), batch_size=CFG.batch_size)

results, trained_models, validation_scores = [], {}, {}
for seed in CFG.seeds:
    model, history = train_one_seed(seed, (train_x, train_y), (val_x, val_y), CFG)
    true, predicted, probabilities = collect_predictions(model, test_loader)
    measured = frame_metrics(true, predicted)
    boundaries = [
        boundary_metrics(t, p, CFG.boundary_tolerance_frames)
        for t, p in zip(true, predicted)
    ]
    scalar = {key: measured[key] for key in ("accuracy", "balanced_accuracy", "macro_f1", "weighted_f1")}
    scalar.update({
        "boundary_precision": float(np.mean([x["boundary_precision"] for x in boundaries])),
        "boundary_recall": float(np.mean([x["boundary_recall"] for x in boundaries])),
        "boundary_mae_frames": float(np.nanmean([
            np.nan if x["boundary_mae_frames"] is None else x["boundary_mae_frames"]
            for x in boundaries
        ])),
        "seed": seed, "method": "temporal_phase_classifier"
    })
    results.append(scalar)
    trained_models[seed] = model.cpu()
    validation_scores[seed] = float(history["val_macro_f1"].max())
    history.to_csv(RUN_DIR / f"training_history_seed_{seed}.csv", index=False)
    torch.save(model.state_dict(), RUN_DIR / f"checkpoint_seed_{seed}.pt")
    with open(RUN_DIR / f"test_report_seed_{seed}.json", "w") as handle:
        json.dump(measured, handle, indent=2)

baseline = majority_baseline(train_y, test_y.shape)
baseline_metrics = frame_metrics(test_y, baseline)
results.append({
    **{key: baseline_metrics[key] for key in ("accuracy", "balanced_accuracy", "macro_f1", "weighted_f1")},
    "boundary_precision": 0.0, "boundary_recall": 0.0,
    "boundary_mae_frames": np.nan, "seed": None, "method": "majority"
})
result_table = pd.DataFrame(results)
display(result_table)
result_table.to_csv(RUN_DIR / "metrics_by_run.csv", index=False)
summary = result_table[result_table.method == "temporal_phase_classifier"].select_dtypes("number").agg(["mean","std"])
display(summary)
summary.to_csv(RUN_DIR / "metrics_summary.csv")
test_rows.to_csv(RUN_DIR / "test_windows.csv", index=False)

In [ ]:
# Select the deployment checkpoint using validation only, never test scores.
best_seed = max(validation_scores, key=validation_scores.get)
final_model = trained_models[best_seed].to(DEVICE)
true, predicted, _ = collect_predictions(final_model, test_loader)
matrix = confusion_matrix(true.ravel(), predicted.ravel(), labels=np.arange(len(PHASES)))
plt.figure(figsize=(10,8))
sns.heatmap(matrix, annot=True, fmt="d", xticklabels=PHASES, yticklabels=PHASES, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Held-out phase confusion matrix")
plt.tight_layout(); plt.savefig(RUN_DIR / "confusion_matrix.png", dpi=200); plt.show()

## 8. ONNX parity and CPU latency

The exact ONNX file exported here must be used for browser evaluation.
Measure browser median/p95 latency separately on the research laptop.

In [ ]:
import onnx
import onnxruntime as ort

onnx_path = RUN_DIR / "temporal_phase_classifier.onnx"
final_model = final_model.cpu().eval()
example = torch.as_tensor(test_x[:1], dtype=torch.float32)
torch.onnx.export(
    final_model, example, onnx_path,
    input_names=["landmarks"], output_names=["phase_logits"],
    dynamic_axes={"landmarks": {0: "batch"}, "phase_logits": {0: "batch"}},
    opset_version=17
)
onnx.checker.check_model(onnx.load(onnx_path))
runtime = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
reference = final_model(example).detach().numpy()
deployed = runtime.run(None, {"landmarks": example.numpy()})[0]
max_error = float(np.max(np.abs(reference - deployed)))
labels_equal = bool(np.array_equal(reference.argmax(-1), deployed.argmax(-1)))
latencies = []
for sample in test_x[:min(100, len(test_x))]:
    start = time.perf_counter()
    runtime.run(None, {"landmarks": sample[None].astype(np.float32)})
    latencies.append((time.perf_counter() - start) * 1000)
parity = {
    "max_abs_logit_error": max_error,
    "predicted_labels_equal": labels_equal,
    "passed_at_1e-4": bool(max_error < 1e-4 and labels_equal),
    "cpu_latency_median_ms": float(np.median(latencies)),
    "cpu_latency_p95_ms": float(np.percentile(latencies, 95)),
    "onnx_sha256": sha256_file(onnx_path),
}
print(parity)
with open(RUN_DIR / "onnx_parity_latency.json", "w") as handle:
    json.dump(parity, handle, indent=2)

## 9. Metadata, provenance, and archive

The metadata records evaluation origin explicitly so synthetic pipeline
checks cannot later be confused with human-verified test performance.

In [ ]:
metadata = {
    "model_type": "temporal-phase-classifier",
    "technique_id": CFG.technique_id,
    "input": {
        "frames": CFG.window, "landmarks": CFG.joints,
        "features": ["x", "y", "z", "visibility"]
    },
    "phase_labels": PHASES,
    "evaluation_origin": "human_verified_grouped_test_split",
    "run_id": RUN_ID,
    "group_level": GROUP_LEVEL,
    "dataset_sha256": sha256_file(CFG.data_path),
    "onnx_sha256": sha256_file(onnx_path),
    "configuration": asdict(CFG),
    "split_indices": {key: value.tolist() for key, value in split_indices.items()},
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "sklearn": sklearn.__version__,
        "onnxruntime": ort.__version__,
    },
    "limitations": [
        "Coverage is limited to documented techniques and participants.",
        "Three-participant system study is a feasibility pilot.",
        "Repetition decoding must be evaluated in the deployed system."
    ],
}
with open(RUN_DIR / "model_metadata.json", "w") as handle:
    json.dump(metadata, handle, indent=2)

import shutil
archive = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
print("Archive:", archive)
from google.colab import files
files.download(archive)